Study on Market Value ionstead of Urban Density

In [ ]:
# 3. Aggregazione Vettorializzata
# .size() è più veloce di .count() perché non controlla i valori nulli colonna per colonna
market_volume = data_df.groupby('postal_code').size().reset_index(name='volume')

print(f"Dataset pulito. Totale CAP unici: {len(market_volume)}")
print(market_volume.head())
print(data_df['postal_code'].dtypes)

In [ ]:
# 1. Caricamento GeoJSON (usiamo il file dei punti)
gdf = gpd.read_file(PC_GEOJSON_PATH)

# Assicuriamoci che la colonna chiave del GeoJSON sia stringa per il merge
gdf['column_1'] = gdf['column_1'].astype(str)

# 2. Merge (Left join per mantenere tutte le tue aree di mercato)
# 'postal_code' del DF vs 'column_1' del GeoJSON
gdf_merged = gdf.merge(market_volume, left_on='column_1', right_on='postal_code', how='left')

# Riempimento dei volumi mancanti con 0 (aree senza proprietà trovate)
gdf_merged['volume'] = gdf_merged['volume'].fillna(0)

Urban density elements inspection

In [ ]:
import pandas as pd
import json

# --- FASE 1: Estrazione dati spaziali dal GeoJSON ---
# Assicurati che il file si trovi nella stessa cartella dello script
with open(PC_GEOJSON_PATH, 'r') as f:
    geo_data = json.load(f)

geo_list = []
for feature in geo_data['features']:
    props = feature['properties']
    geo_list.append({
        'postal_code': str(props.get('column_1')), # Convertito in stringa per il join
        'longitude': props.get('column_3'),
        'latitude': props.get('column_4')
    })

# Creiamo il dataframe di lookup spaziale
df_geo = pd.DataFrame(geo_list)

# --- FASE 2: Preparazione dati dello scraper ---
# Carica il tuo dataset (sostituisci col nome reale del tuo dataframe/CSV)
df_scraper = pd.read_csv(CLEANED_CSV_PATH)
df_scraper['postal_code'] = df_scraper['postal_code'].astype(str)

# Calcoliamo il prezzo al mq per ogni singolo annuncio (se non l'hai già fatto)
df_scraper['price_per_m2'] = df_scraper['price'] / df_scraper['living_area_m2']

# --- FASE 3: Aggregazione per Densità e Valore ---
# Raggruppiamo per codice postale calcolando la densità (conteggio) e il valore (media)
df_grouped = df_scraper.groupby('postal_code').agg(
    property_count=('postal_code', 'count'),       # Questa è la nostra DENSITÀ
    avg_price_m2=('price_per_m2', 'mean')       # Questo è il nostro VALORE DI MERCATO
).reset_index()

# --- FASE 4: Join e Pulizia ---
# Uniamo le coordinate geografiche alle metriche calcolate
df_final = pd.merge(df_grouped, df_geo, on='postal_code', how='inner')

# Rimuoviamo eventuali righe senza coordinate o prezzo
df_final = df_final.dropna(subset=['latitude', 'longitude'])

# Riduci il dataset a un'unica riga per codice postale
df_final_clean = df_final.drop_duplicates(subset=['postal_code'])

# Esportiamo il dataset pronto e pulito per Superset
df_final_clean.to_csv(GEO_CSV_PATH, index=False)
print("Dataset creato con successo!")

- **Strategy 1**: Before passing the DataFrame to Seaborn, apply a quick filter to exclude the upper end of the price range (e.g., keep only data below the 95th or 98th percentile).
- **Strategy 2**: Pass an explicit ordered list (e.g., ['Low Density', 'Medium Density', 'High Density']) to the `size_order` argument to force urban centers to appear visually larger than rural areas.
- **Strategy 3**: Remember to set the plot's aspect ratio to 'equal' or calculate a figure size proportional to the actual differences between Belgium's latitude and longitude to maintain geographic accuracy.

In [ ]:
# ============ PANEL 1 - THE MAP ============
# strategy 1
q_98 = data_df['price_per_m2'].quantile(0.98)
plot_df = data_df[data_df['price_per_m2'] <= q_98].copy()

# assegnation of a numeric score for thermic map of urban density in belgium
plot_df['density_score'] = 1    # bt default
plot_df.loc[
    plot_df['urban_density'].str.contains("Medium", na=False), 'density_score'
] = 2
plot_df.loc[
    plot_df['urban_density'].str.contains("High", na=False), 'density_score'
]= 3

# upload geopoliticals borders of Belgium
be_map = gpd.read_file(BE_GEOJSON_PATH)

# Plot configuration
fig, (ax_map, ax_stat) = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1.3, 1]})

cmap_prices = LinearSegmentedColormap.from_list(
    "blue_white_gold", ["#1a4b8c", "#ffffff", "#b8860b"]
)

scatter = ax_map.scatter(  # <--- Cambiato ax in ax_map
    plot_df['longitude'],
    plot_df['latitude'],
    c=plot_df['price_per_m2'],
    s=10,
    cmap=cmap_prices,
    alpha=0.75,
    edgecolors="none"
)

be_map.plot(ax=ax_map, color="#f5f5f5", edgecolor="#e0e0e0", linewidth=1.0)

scatter = ax_map.scatter(
    plot_df['longitude'],
    plot_df['latitude'],
    c=plot_df['price_per_m2'],
    s=10,
    cmap=cmap_prices,   # custom palette declared above
    alpha=0.75,
    edgecolors="none"
)

be_map.plot(ax=ax_map, color="none", edgecolor="#4a4a4a", linewidth=1.0)

ax_map.set_axis_off()
ax_map.set_aspect("equal", adjustable="box")

divider = make_axes_locatable(ax_map)
cax_right = divider.append_axes("right", size="3%", pad=0.2)
# bar aesthetic configuration
cbar_right = fig.colorbar(scatter, cax=cax_right)
cbar_right.set_label("Price per m² (€)", fontsize=11, fontweight="bold")

# ============ PANEL 2 - THE BOXPLOT ============
# prices distribution per urban density category
sns.boxplot(
    data=plot_df,
    x='urban_density',
    y='price_per_m2',
    order=['Low Density / Rural Areas', 'Medium Density Suburbs', 'High Urban Density Hubs'], # Ordine logico crescente
    palette=["#e6f2ff", "#99c2ff", "#1a4b8c"], # Palette discreta coerente (da azzurro a blu scuro)
    ax=ax_stat,
    fliersize=2,
    linewidth=1.2
)

ax_stat.grid(axis="y", linestyle="--", alpha=0.5)
ax_stat.set_title("Price Distribution by Urban Density", fontsize=12, fontweight="bold", pad=10)
ax_stat.set_xlabel("Urban Density Category", fontsize=11, fontweight="bold")
ax_stat.set_ylabel("Price per m² (€)", fontsize=11, fontweight="bold")

# ============ GOLBAL TITLE AND SAVING ============

fig.suptitle(
    "Belgium Real Estate Market Analysis\nGeographical Hotspots vs Statistical Urban Density Impact",
    fontsize=16,
    fontweight="bold",
    x=0.5,
    y=0.96
)

plt.subplots_adjust(top=0.85, bottom=0.1, left=0.05, right=0.95, wspace=0.25)

plt.savefig(
    f"{PLOTS_PATH}Belgium_Market_Map_with_Urban_Density.png", 
    bbox_inches='tight', 
    dpi=330
)

plt.show()
plt.close()

1. La trappola degli Outliers (Scala dei colori)
Se nel tuo dataset ci sono immobili con un price_per_m2 astronomico (es. 15.000€/m²), la palette di colori del hue si schiaccerà: vedrai un solo punto accesissimo e tutti gli altri dello stesso colore neutro, annullando l'insight.

Strategia: Prima di passare il DataFrame a Seaborn, applica un filtro rapido per escludere l'estremo superiore dei prezzi (es. tieni solo i dati sotto il 95° o 98° percentile).

2. La trappola della colonna urban_density
La feature urban_density nel tuo dataset contiene stringhe (es. 'High Density', 'Low Density'). Se la passi direttamente all'argomento size, Seaborn assegnerà le dimensioni dei punti in base all'ordine alfabetico o di apparizione, il che è concettualmente errato.

Strategia: Passa all'argomento size_order una lista ordinata esplicita (es. ['Low Density', 'Medium Density', 'High Density']) per forzare i centri urbani a essere visivamente più grandi delle aree rurali.

3. Il fattore di distorsione geometrica (Aspect Ratio)
Se lasci che Matplotlib decida le proporzioni della figura in automatico, la mappa del Belgio apparirà schiacciata in verticale o allungata in orizzontale.

Strategia: Ricordati di impostare l'aspect ratio del grafico a equal o calcola una figsize proporzionata alle reali differenze tra latitudine e longitudine del Belgio per mantenere la coerenza geografica.

*Se vuoi mappare gli "Hotspots" intorno a Bruxelles, Anversa e Gand, la logica delle tue coppie condizionali non serve. Ti servono solo latitude, longitude, price_per_m2 e urban_density.*

*L'idea visiva: Un grafico a dispersione (sns.scatterplot) dove l'asse X e Y sono le coordinate geografiche, il colore (hue) rappresenta il prezzo al metro quadro, e la dimensione dei punti (size) indica la densità urbana.*

*Cosa cercare: I cluster di punti con prezzo basso situati immediatamente fuori dalle bolle ad alta densità (le grandi città).*

Other conditional pairs:
- 'parking_count'/'property_type' - if 'property_type' is NOT House, 'parking_count' COULD be 0
- 'parking_count'/'has_garage' - if 'has_garage' is 0, 'parking_count' COULD be 0
- 'parking_count'/'urban_density' - if 'urban_density' is 'High Density (City Center/Brussels)' or 'High Density (Urban Hub)', 'parking_count' COULD be 0
- 'floor_number'/'property_type' - if 'property_type' is 'House', 'floor_number' should be NA (not_applicable)
- 'floors_total'/'property_type'  
- 'facades'/'property_type'
- 'building_year'/'state_of_the_building'
- 'kitchen_equipped'/?
